[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/tilings.ipynb)

# Tilings of Three Geometries

This example tiles the sphere, the Euclidean plane and the hyperbolic plane with a single reflection construction. Three mirrors meeting at a half turn divided by two, three and a third integer cut out a triangle whose angles sum to more than a half turn at five, to a half turn at six, and to less at seven, so repeated reflections in those three mirrors tile the sphere, the flat plane and the hyperbolic plane with the angle sum alone selecting the geometry.

Each geometry is modelled in a three-generator plane-based algebra where the angle sum fixes the metric square of the third basis plane `normal` to plus one, zero or minus one. Mirrors are extrinsic planes `Plane` and points are intrinsic points `Point`, so both the metric-free half-space test `mirror & point` and the reflection sandwich `mirror >> point` carry identical algebraic expressions across all three signatures, including the degenerate flat plane. Exponentiating the bivector `((x ^ normal) * (-distance / 2)).exp()` rotates the sphere, translates the flat plane and boosts the hyperbolic plane along one shared code path.

In split Riemannian, Euclidean and Lorentzian coordinate formulas for Coxeter groups $[5, 3]$, $[6, 3]$ and $[7, 3]$, spherical reflections use $\mathrm{O}(3)$ Householder matrices $I - 2\,\mathbf{n}\mathbf{n}^{\mathsf T}$, planar reflections switch to inhomogeneous affine rules $\mathbf{p} \mapsto \mathbf{p} - 2(\mathbf{n}\cdot\mathbf{p} + d)\mathbf{n}$ and translations $\mathbf{p} + \mathbf{t}$, and hyperbolic reflections on the Poincaré disk require Möbius inversions or Minkowski reflections $I - 2\,\eta\,\mathbf{n}\mathbf{n}^{\mathsf T}\eta / (\mathbf{n}^{\mathsf T}\eta\mathbf{n})$ coupled with $\mathrm{SO}^+(2, 1)$ Lorentz boosts.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_states
from numga.wgsl import Shader, supersampled
from examples.math.reflections import render

## Selecting the signature from the angle sum

Fixing the first two mirrors in the horizontal plane determines the horizontal projection of the third mirror from its two dihedral cosines. When the triangle's angles exceed a half turn that horizontal projection has squared length below one, requiring a positive-square third basis plane to complete a unit mirror; when the angles equal a half turn its length is already one, requiring a null third basis plane whose coefficient sets the Euclidean tile scale; and when the angles fall short of a half turn its squared length exceeds one, requiring a negative-square third basis plane.

In [ ]:
# Dihedral angles of the three fundamental triangles and signatures keyed by curvature sign.
first_second = np.pi / 2
first_third = np.pi / 3
second_thirds = np.pi / np.array([5, 6, 7])
signatures = {1: "x+y+z+", 0: "x+y+z0", -1: "x+y+z-"}
flat_offset = 0.5

In [ ]:
def triangle_geometry(second_third: float):
    """Selects the algebra signature from the angle sum and constructs the multivector factory, curvature sign, third basis plane, and three mirror planes (`[3] Plane+`)."""
    curvature = int(np.sign(np.round(first_second + first_third + second_third - np.pi, 12)))
    mv = NumpyContext(Algebra(signatures[curvature])).multivector
    first = mv.x                                                                # [] Plane+
    first_turn = (mv.xy * (-(np.pi - first_second) / 2)).exp()                  # [] Rotor+
    second = first_turn >> first                                                # [] Plane+
    # Reciprocal vectors in `plane` fix the horizontal part; `normal.inverse()` supplies the signed complement.
    plane = first ^ second                                                      # [] Bivector+
    in_plane = -np.cos(first_third) * (second | plane.inverse()) - np.cos(second_third) * (plane.inverse() | first)   # [] Plane+
    normal = mv.z                                                               # [] Plane+
    third = in_plane + (((1 - in_plane.squared()) / curvature).square_root() * normal.inverse() if curvature
                        else flat_offset * normal)                              # [] Plane+
    return mv, curvature, normal, stack([first, second, third])


geometries = [triangle_geometry(angle) for angle in second_thirds]

## Conformal disk mapping and WGSL folding shader

One formula in `normal.squared()` maps the unit disk conformally to all three spaces at once: stereographically onto the sphere, to scale onto the Euclidean plane, and as the Poincaré disk onto the hyperbolic plane. Compiling one WGSL shader source against each algebra folds every supersampled pixel across the three mirrors by testing `mirrors[index] & point` and applying `mirrors[index] >> point`.

In [ ]:
# Pixel resolution per disk and reflection rounds per sample.
n_pixels = 160
n_rounds = 20

In [ ]:
fold_shader_source = render.tiling_colouring() + (
    'alias Sample = vec3<f32>;\n'
    'var<uniform> mirrors: array<Plane, 3>;\n'
    'var<uniform> normal: Plane;\n'
    'var<uniform> motion: Rotor;\n\n'
    'fn sample(place: vec2<f32>) -> Sample {\n'
    '    let at = screen(place);\n'
    '    let squared_distance = dot(at, at);\n'
    '    if (squared_distance >= 1) {\n'
    '        return background;\n'
    '    }\n'
    '    // Conformal lift from the unit disk into spherical, Euclidean, or hyperbolic space.\n'
    '    let lifted = mv.x.dual() * (2 * at.x) + mv.y.dual() * (2 * at.y)\n'
    '                 + normal.dual() * (1 - normal.squared() * squared_distance);\n'
    '    var point = motion >> lifted;\n'
    '    var flips = 0u;\n'
    '    for (var round = 0u; round < n_rounds; round++) {\n'
    '        for (var index = 0u; index < 3u; index++) {\n'
    '            if ((mirrors[index] & point) < 0) {\n'
    '                point = mirrors[index] >> point;\n'
    '                flips++;\n'
    '            }\n'
    '        }\n'
    '    }\n'
    '    var nearest = abs(mirrors[0] & point);\n'
    '    var total = nearest;\n'
    '    for (var index = 1u; index < 3u; index++) {\n'
    '        let side = abs(mirrors[index] & point);\n'
    '        nearest = min(nearest, side);\n'
    '        total += side;\n'
    '    }\n'
    '    return tile(flips, nearest / total);\n'
    '}\n'
)


def tiling_shader(algebra: Algebra) -> Shader:
    """Compiles the supersampled folding shader for `algebra` with `Plane` and `Rotor` uniforms."""
    return Shader(
        supersampled() + fold_shader_source, algebra,
        dict(Plane=algebra.gatype.vector(), Rotor=algebra.gatype.rotor()),
        render.tiling_colours | dict(n_rounds=n_rounds),
    )

## Motion across the three tilings

Exponentiating the bivector wedged from the horizontal axis and `normal` drives all three views horizontally, turning the sphere by an angle, translating the flat plane by a distance, and boosting the hyperbolic plane by a rapidity.

In [ ]:
# Frame count, frame duration, and total travel parameter.
n_frames = 90
frame_duration_ms = 40
travel = 1.2

In [ ]:
shaders = [tiling_shader(mv.algebra) for mv, _, _, _ in geometries]


def tiling_frames(n_frames: int) -> Iterator[list[np.ndarray]]:
    """Yields the three rendered disk images across `n_frames` steps of uniform travel."""
    for distance in np.linspace(0.0, travel, n_frames):
        yield [
            shader(
                (n_pixels, n_pixels), mirrors=mirrors, normal=normal,
                motion=((mv.x ^ normal) * (-distance / 2)).exp(),
            )["image"]
            for shader, (mv, _, normal, mirrors) in zip(shaders, geometries)
        ]


recorded_tilings = show_states(
    tiling_frames(n_frames), render.tilings_animation,
    name="tilings", duration_ms=frame_duration_ms, n_frames=n_frames,
);

From left to right, the stereographic sphere displays pentagonal clusters of ten triangles, the flat plane glides rigidly across hexagonal rings of twelve triangles, and the Poincaré disk sweeps heptagonal rings of fourteen triangles that contract toward the boundary circle.

In [ ]:
# checks
assert [curvature for _, curvature, _, _ in geometries] == [1, 0, -1]
assert len(recorded_tilings) == n_frames
for (mv, curvature, normal, mirrors), angle in zip(geometries, second_thirds):
    assert (((mv.x ^ normal).squared() + curvature).abs() < 1e-10).all()
    first, second, third = mirrors[0], mirrors[1], mirrors[2]
    for one, other, between in ((first, second, first_second), (first, third, first_third), (second, third, angle)):
        assert (((one | other) + np.cos(between)).abs() < 1e-10).all()
    origin_point = normal.dual()                                                # [] Point-
    motion = ((mv.x ^ normal) * (-travel / 2)).exp()                            # [] Rotor+
    folded_point = motion >> origin_point                                       # [] Point-
    for _ in range(n_rounds):
        for mirror in mirrors:
            if (mirror & folded_point) < 0:
                folded_point = mirror >> folded_point                           # [] Point-
    sides = mirrors & folded_point                                              # [3] Scalar+
    assert sides.isfinite().all() and (sides > -1e-9).all()